In [1]:
import pandas as pd
import numpy as np

TARGETS = ["x"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_x,MSE_ZZx1_x,R2_ZZx2_x,MSE_ZZx2_x,...,R2_LSG_1_x,MSE_LSG_1_x,R2_LSG_2_x,MSE_LSG_2_x,R2_ZZx1_inv_x,MSE_ZZx1_inv_x,R2_zzx2_inv2_x,MSE_zzx2_inv2_x,R2_semiCirc_x,MSE_semiCirc_x
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed9823,[1],0.5,0.5,0.01,9823,0.192857,0.786238,-5.945147,0.701736,...,-29.501372,-0.855211,0.566587,0.604849,-4.368314,0.309784,-31.794291,0.137282,-4.883108,-2.881149
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed7935,[1],0.5,0.5,0.01,7935,0.466453,0.854746,-6.841502,0.694591,...,-15.381369,-0.400419,0.807653,0.807346,-0.136047,0.747403,-41.853195,0.146167,0.026893,0.049428
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed3847,[1],0.5,0.5,0.01,3847,0.241493,0.798737,-5.788695,0.706937,...,-27.001884,-0.727045,0.660383,0.724988,-2.615731,0.491272,-32.466692,0.182296,-1.377781,-0.488374
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed1434,[1],0.5,0.5,0.01,1434,0.394564,0.835109,-5.831976,0.709713,...,-14.170296,-0.322553,0.760017,0.786140,-0.241261,0.717665,-37.065472,0.197827,0.054743,0.081077
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed4683,[1],0.5,0.5,0.01,4683,0.282729,0.808083,-7.125618,0.692941,...,-28.292351,-0.877295,0.650198,0.701752,-3.276623,0.430202,-38.188678,0.137874,-2.394382,-1.178316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1930,model_arch58_r0.01_Ld0.7_Lp0.3_seed9807,[58],0.7,0.3,0.01,9807,0.955373,0.970441,-1.515251,0.823226,...,-2.790244,-1.035984,-0.785637,0.784486,-1.265069,0.870380,-3.540248,0.676823,-0.382952,-0.217789
1931,model_arch58_r0.01_Ld0.7_Lp0.3_seed2551,[58],0.7,0.3,0.01,2551,-0.220028,0.711578,-1.255498,0.776917,...,-0.186683,-0.289932,0.445080,0.690263,0.261224,0.697255,-1.723455,0.701930,0.895798,0.410343
1932,model_arch58_r0.9_Ld0.7_Lp0.3_seed2402,[58],0.7,0.3,0.90,2402,0.960497,0.971317,-3.860488,0.796199,...,-8.405282,-1.905919,-1.239492,0.737044,-2.379521,0.851923,-14.670644,0.474729,-1.978635,-0.852142
1933,model_arch58_r0.9_Ld0.7_Lp0.3_seed1841,[58],0.7,0.3,0.90,1841,0.965172,0.970046,-2.570730,0.782005,...,-11.088895,-2.355826,-1.146270,0.735867,-2.347596,0.857179,-22.147103,0.373276,-2.838862,-1.167383


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - x


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
382,model_arch13_r0.01_Ld0.7_Lp0.3_seed3847,[13],0.942482,0.622943,-0.105806,0.384931
401,model_arch14_r0.01_Ld0.3_Lp0.7_seed7935,[14],0.961980,0.292179,0.010430,0.329002
574,model_arch20_r0.01_Ld0.5_Lp0.5_seed4683,[20],0.927219,0.634882,-0.182632,0.323671
418,model_arch14_r0.9_Ld0.7_Lp0.3_seed1434,[14],0.942274,0.669760,-0.290621,0.307659
314,model_arch11_r0.01_Ld0.3_Lp0.7_seed4683,[11],0.969602,0.188075,0.008137,0.306686



📊 MÉTRICAS COMPLETAS - TOP 5 (x)


,model,Neurons,R2_ZZx1_x,R2_ZZx2_x,R2_ZZxReto_x,R2_ZZy1_x,R2_ZZy2_x,R2_LSG_1_x,R2_LSG_2_x,R2_ZZx1_inv_x,R2_semiCirc_x,R2_train_mean,R2_val_mean,R2_test_mean,Score
382,model_arch13_r0.01_Ld0.7_Lp0.3_seed3847,[13],0.942482,0.622943,0.815433,-0.955152,-1.412625,-1.156166,0.447997,0.656188,0.863685,0.942482,0.622943,-0.105806,0.384931
401,model_arch14_r0.01_Ld0.3_Lp0.7_seed7935,[14],0.961980,0.292179,0.828679,-1.629926,-0.890966,-0.054965,0.304166,0.612453,0.903567,0.961980,0.292179,0.010430,0.329002
574,model_arch20_r0.01_Ld0.5_Lp0.5_seed4683,[20],0.927219,0.634882,0.911184,-3.043791,-0.978027,-0.195932,0.403663,0.714794,0.909684,0.927219,0.634882,-0.182632,0.323671
418,model_arch14_r0.9_Ld0.7_Lp0.3_seed1434,[14],0.942274,0.669760,0.819243,-2.802589,-1.309961,-0.533106,0.210647,0.627290,0.954130,0.942274,0.669760,-0.290621,0.307659
314,model_arch11_r0.01_Ld0.3_Lp0.7_seed4683,[11],0.969602,0.188075,0.830019,-0.711496,-1.181364,-0.541986,0.172711,0.609043,0.880031,0.969602,0.188075,0.008137,0.306686


In [5]:
final_table.to_excel("BestModels-otm.xlsx")